setup

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import SEED, LABELS, load_narratives, build_dataset, make_splits

np.random.seed(SEED)
ZIP_PATH = ROOT / "data/raw/CCDB_Export_7_August_2024_through_October_2024.zip"
CACHE = ROOT / "data/processed/narratives_aug_oct_2024.csv"
PROC = ROOT / "data/processed"
REPORTS = ROOT / "reports"
print(ROOT)

e:\complaint-classifier


Load

In [2]:
raw = load_narratives(ZIP_PATH, CACHE)
assert len(raw) == 229_944, len(raw)
raw.shape

(229944, 5)

build the dataset; must reproduce the EDA plan

In [3]:
data, report = build_dataset(raw)
for step, n in report.items():
    print(f"{step:>24}: {n:,}")

assert report["after_template_dedup"] == 87_055, report
assert len(data) == 56_783, len(data)
data["label"].value_counts().loc[LABELS]

                raw_rows: 229,944
   after_label_conflicts: 219,428
    after_template_dedup: 87,055
    after_product_filter: 84,701
              final_rows: 56,783


label
Credit reporting               20000
Debt collection                12108
Credit card                     7622
Checking or savings account     7502
Mortgage                        3121
Money transfer                  2573
Student loan                    1991
Vehicle loan or lease           1866
Name: count, dtype: int64

very short complaints

In [4]:
n_words = data["text"].str.split().str.len()
print("Complaints under 10 words:", int((n_words < 10).sum()), "of", len(data))
print((n_words < 10).groupby(data["label"]).agg(["sum", "mean"]).round(3))
print()
for t in data.loc[n_words < 10, "text"].head(8):
    print("-", t)

Complaints under 10 words: 257 of 56783
                             sum   mean
label                                  
Checking or savings account    8  0.001
Credit card                   10  0.001
Credit reporting             168  0.008
Debt collection               48  0.004
Money transfer                 6  0.002
Mortgage                       2  0.001
Student loan                   6  0.003
Vehicle loan or lease          9  0.005

- I did not authorize [REDACTED] [DATE], [REDACTED]/[REDACTED]/[REDACTED]
- Noticed some inquiries that aren't mine, they are fraudulent.
- In accurate reporting from all Three Credit bureau ......
- Dignify [REDACTED], [REDACTED]
- I have identified unauthorized inquiries on my credit report.
- [REDACTED] [DATE] [REDACTED] [DATE] [REDACTED], [REDACTED]
- My Credit was ran without my knowledge or permission.
- There are unauthorized inquiries on my credit report.


**Finding:** Only 257 of 56,783 complaints (0.5%) have fewer than 10 words, and most of them are credit reporting (168). Some are just redaction placeholders, like "[REDACTED] [DATE] [REDACTED]", with no information at all.

**What it means:** A complaint with no real words cannot be classified by anyone, so it would only add noise to the test score. I will drop complaints with fewer than 5 real words (placeholders excluded).

split

In [5]:
train, val, test = make_splits(data)
print(len(train), len(val), len(test))

counts = pd.DataFrame({"train": train["label"].value_counts(),
                       "val": val["label"].value_counts(),
                       "test": test["label"].value_counts()}).loc[LABELS]
display(counts)
display((counts / counts.sum() * 100).round(2))

assert data["complaint_id"].is_unique
assert not (set(train["complaint_id"]) & set(val["complaint_id"]))
assert not (set(train["complaint_id"]) & set(test["complaint_id"]))
assert not (set(val["complaint_id"]) & set(test["complaint_id"]))

39747 8518 8518


,train,val,test
label,,,
Credit reporting,14000,3000,3000
Debt collection,8476,1816,1816
Credit card,5335,1143,1144
Checking or savings account,5251,1126,1125
Mortgage,2185,468,468
Money transfer,1801,386,386
Student loan,1393,299,299
Vehicle loan or lease,1306,280,280


,train,val,test
label,,,
Credit reporting,35.22,35.22,35.22
Debt collection,21.32,21.32,21.32
Credit card,13.42,13.42,13.43
Checking or savings account,13.21,13.22,13.21
Mortgage,5.50,5.49,5.49
Money transfer,4.53,4.53,4.53
Student loan,3.50,3.51,3.51
Vehicle loan or lease,3.29,3.29,3.29


**Result:** The first stratified split (70/15/15) gives 39,747 / 8,518 / 8,518 complaints, and every class has exactly the same percentage in train, validation and test. No complaint ID appears in more than one split. This is version 1 of the split. The leakage check below shows it still has a problem, so it gets replaced further down.

save; the CSVs stay gitignored

In [6]:
train.to_csv(PROC / "train.csv", index=False)
val.to_csv(PROC / "val.csv", index=False)
test.to_csv(PROC / "test.csv", index=False)
counts.assign(total=counts.sum(axis=1)).to_csv(REPORTS / "split_summary.csv")
print([p.name for p in sorted(PROC.glob("*.csv"))])

['narratives_aug_oct_2024.csv', 'test.csv', 'train.csv', 'val.csv']


leakage check: how similar is each test complaint to its closest training complaint?

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.5, max_features=100_000,
                      sublinear_tf=True, stop_words="english")
X_train = vec.fit_transform(train["text"])        # fitted on TRAIN only
X_val = vec.transform(val["text"])
X_test = vec.transform(test["text"])


def max_similarity(queries, reference, chunk=500):
    """Cosine similarity of each query to its nearest reference document."""
    out = np.zeros(queries.shape[0])
    for i in range(0, queries.shape[0], chunk):
        sims = queries[i:i + chunk] @ reference.T
        out[i:i + chunk] = sims.max(axis=1).toarray().ravel()
    return out


val_sim = max_similarity(X_val, X_train)
test_sim = max_similarity(X_test, X_train)

for name, s in [("val", val_sim), ("test", test_sim)]:
    shares = {f">={t}": f"{(s >= t).mean():.1%}" for t in (0.95, 0.90, 0.80)}
    print(f"{name}: median {np.median(s):.3f} | {shares}")

val: median 0.232 | {'>=0.95': '6.7%', '>=0.9': '10.9%', '>=0.8': '16.1%'}
test: median 0.233 | {'>=0.95': '6.1%', '>=0.9': '10.2%', '>=0.8': '15.5%'}


**Finding:** For each test complaint I found its most similar training complaint (TF-IDF cosine similarity). The median similarity is 0.23, which is healthy, but 10.2% of test complaints have a near-identical twin in train (similarity 0.90 or more) and 15.5% have one at 0.80 or more.

**What it means:** My rule was that more than 5% means I must fix it before training. A model could score well on these complaints just by remembering the training letter, so the scores would look better than the model really is.

which classes are affected, plus a look at real pairs

In [8]:
high = test_sim >= 0.90
print("Share of test complaints with a >=0.90 twin in train, by class (%):")
print(pd.Series(high).groupby(test["label"].values).mean().mul(100).round(1)
        .sort_values(ascending=False).to_string())

idx = np.where(high)[0]
rng = np.random.default_rng(SEED)
for i in rng.choice(idx, size=min(3, len(idx)), replace=False):
    j = (X_test[i] @ X_train.T).toarray().argmax()
    print("\n" + "=" * 80)
    print(f"similarity {test_sim[i]:.3f} | test label: {test['label'][i]} | train label: {train['label'][j]}")
    print("TEST :", test["text"][i][:250])
    print("TRAIN:", train["text"][j][:250])

Share of test complaints with a >=0.90 twin in train, by class (%):
Credit reporting               19.9
Debt collection                12.9
Vehicle loan or lease           2.1
Money transfer                  1.8
Mortgage                        1.3
Credit card                     1.2
Checking or savings account     0.7
Student loan                    0.0

similarity 0.906 | test label: Credit reporting | train label: Credit reporting
TEST : Please note that this is my THIRD WRITTEN REQUEST. I have contacted the CREDITORS BELOW and they have no proof that I initiated such inquiries, no written authorization, and have provided no permissible purpose. Since they have no authorization from 
TRAIN: Please note that this is my SECOND WRITTEN REQUEST. I have been contacted the CREDITORS BELOW AND THEY HAVE NO PROOF THAT I INITIATED SUCH INQUIRIES, NO WRITTEN AUTHORIZATION, and have provided no permissible purpose. Since they have no authorization

similarity 0.922 | test label: Credit reportin

**Finding:** The twins are concentrated in credit reporting (19.9% of its test complaints) and debt collection (12.9%); other classes are at 2.1% or lower. The sample pairs are the same letter with small edits, for example "SECOND WRITTEN REQUEST" in train and "THIRD WRITTEN REQUEST" in test.

**What it means:** My earlier check on the first 150 characters missed these, because the openings differ slightly. I need to compare whole texts by similarity, not by exact match.

keep the “before” numbers, and look at pairs in the 0.80 to 0.90 band

In [9]:
before = {"val": val_sim.copy(), "test": test_sim.copy()}

band = np.where((test_sim >= 0.80) & (test_sim < 0.90))[0]
print(f"Test complaints whose closest train twin is in the 0.80-0.90 band: "
      f"{len(band)} ({len(band) / len(test):.1%})")
rng = np.random.default_rng(SEED)
for i in rng.choice(band, size=min(4, len(band)), replace=False):
    j = (X_test[i] @ X_train.T).toarray().argmax()
    print("\n" + "=" * 80)
    print(f"similarity {test_sim[i]:.3f} | test: {test['label'][i]} | train: {train['label'][j]}")
    print("TEST :", test["text"][i][:250])
    print("TRAIN:", train["text"][j][:250])

Test complaints whose closest train twin is in the 0.80-0.90 band: 450 (5.3%)

similarity 0.859 | test: Credit reporting | train: Credit reporting
TEST : [REDACTED] Subject : Complaint Regarding Inaccurate Late Payment Entry on Credit Report Dear CFPB, I am writing to file a formal complaint against Experian regarding an inaccurate late payment entry on my credit report that has not been resolved desp
TRAIN: [DATE] Subject : Complaint Regarding Inaccurate Late Payment Entry on Credit Report Dear CFPB, I am writing to file a formal complaint against Experian regarding an inaccurate late payment entry on my credit report that has not been resolved despite 

similarity 0.895 | test: Credit reporting | train: Credit reporting
TEST : Background : 1. Vehicle Purchase : I purchased a vehicle from Toyota on [REDACTED]/[REDACTED]/year>. The vehicle experienced a [REDACTED] battery failure the very next day ( [REDACTED]/[REDACTED]/year> ), and I have the receipt for the battery replac
TRAIN: Bac

**Finding:** 5.3% of test complaints have their closest twin in the 0.80 to 0.90 band. Reading four examples, three are clearly the same text with a few words changed (a date placeholder, a redacted brand name, "Collection agency" instead of "[REDACTED]"), and one is a partly rewritten version of the same letter.

**What it means:** The 0.80 threshold removes real copies, not just similar topics, so I will use 0.80 for the near-duplicate removal. This step runs before the split, so no copy can end up on both sides.

remove near-copies and uninformative complaints, then show class sizes before and after

In [10]:
import time
from src.data import NEAR_DUP_THRESHOLD, drop_near_duplicates, drop_uninformative

n0 = len(data)
t0 = time.time()
data2 = drop_near_duplicates(data)
print(f"near-duplicate removal (threshold {NEAR_DUP_THRESHOLD}): {n0:,} -> {len(data2):,} "
      f"in {time.time() - t0:.0f}s")

n1 = len(data2)
data2 = drop_uninformative(data2)
print(f"uninformative removal: {n1:,} -> {len(data2):,}")

sizes = pd.DataFrame({"before": data["label"].value_counts(),
                      "after": data2["label"].value_counts()}).loc[LABELS]
sizes["kept_%"] = (sizes["after"] / sizes["before"] * 100).round(1)
display(sizes)

near-duplicate removal (threshold 0.8): 56,783 -> 50,511 in 67s
uninformative removal: 50,511 -> 50,480


,before,after,kept_%
label,,,
Credit reporting,20000,15670,78.4
Debt collection,12108,10499,86.7
Credit card,7622,7462,97.9
Checking or savings account,7502,7432,99.1
Mortgage,3121,3098,99.3
Money transfer,2573,2511,97.6
Student loan,1991,1977,99.3
Vehicle loan or lease,1866,1831,98.1


**Result:** Removing near-copies (within each class) took the data from 56,783 to 50,511 rows, and removing complaints with fewer than 5 real words took it to 50,480. Credit reporting lost the most (20,000 to 15,670, keeping 78.4%) and debt collection next (12,108 to 10,499, keeping 86.7%); all other classes kept at least 97.6%.

**What it means:** The templated letters were almost all in credit reporting and debt collection. The final dataset has 8 classes and 50,480 complaints.

re-split on the cleaned data and overwrite the saved files

In [11]:
train, val, test = make_splits(data2)
print(len(train), len(val), len(test))

counts = pd.DataFrame({"train": train["label"].value_counts(),
                       "val": val["label"].value_counts(),
                       "test": test["label"].value_counts()}).loc[LABELS]
display(counts)

assert data2["complaint_id"].is_unique
assert not (set(train["complaint_id"]) & set(val["complaint_id"]))
assert not (set(train["complaint_id"]) & set(test["complaint_id"]))
assert not (set(val["complaint_id"]) & set(test["complaint_id"]))

train.to_csv(PROC / "train.csv", index=False)
val.to_csv(PROC / "val.csv", index=False)
test.to_csv(PROC / "test.csv", index=False)
counts.assign(total=counts.sum(axis=1)).to_csv(REPORTS / "split_summary.csv")

35335 7573 7572


,train,val,test
label,,,
Credit reporting,10969,2351,2350
Debt collection,7349,1575,1575
Credit card,5224,1119,1119
Checking or savings account,5202,1115,1115
Mortgage,2168,465,465
Money transfer,1757,377,377
Student loan,1384,297,296
Vehicle loan or lease,1282,274,275


**Result:** The new stratified split has 35,335 train, 7,573 validation and 7,572 test complaints, again with identical class percentages in all three. The smallest test classes are vehicle loan (275) and student loan (296), so their per-class scores will be noisy. These files overwrite the earlier version and are the ones every model will use.

measure leakage again, and save a before/after table

In [12]:
vec2 = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.5, max_features=100_000,
                       sublinear_tf=True, stop_words="english")
X_train2 = vec2.fit_transform(train["text"])            # fitted on the NEW train only
val_sim2 = max_similarity(vec2.transform(val["text"]), X_train2)
test_sim2 = max_similarity(vec2.transform(test["text"]), X_train2)

rows = []
for name, old, new in [("val", before["val"], val_sim2), ("test", before["test"], test_sim2)]:
    for t in (0.95, 0.90, 0.80):
        rows.append({"split": name, "similarity>=": t,
                     "before_%": round((old >= t).mean() * 100, 1),
                     "after_%": round((new >= t).mean() * 100, 1)})
leak = pd.DataFrame(rows)
display(leak)
leak.to_csv(REPORTS / "leakage_check.csv", index=False)
print("median similarity after:", round(float(np.median(test_sim2)), 3))

,split,similarity>=,before_%,after_%
0,val,0.95,6.7,0.1
1,val,0.90,10.9,0.3
2,val,0.80,16.1,2.0
3,test,0.95,6.1,0.1
4,test,0.90,10.2,0.3
5,test,0.80,15.5,1.6


median similarity after: 0.211


**Result:** After near-duplicate removal, the share of test complaints with a twin in train dropped from 6.1% to 0.1% at similarity 0.95 or more, from 10.2% to 0.3% at 0.90 or more, and from 15.5% to 1.6% at 0.80 or more. Validation shows the same pattern. The median similarity went from 0.233 to 0.211.

**What it means:** The 0.3% left is far below my 2% limit, so I accept this split. The leakage was not completely zero because the similarity measure in this check is fitted on the new training set only, so small differences remain.

look at the closest remaining pairs

In [13]:
Xt = vec2.transform(test["text"])
for i in np.argsort(test_sim2)[::-1][:3]:
    j = (Xt[i] @ X_train2.T).toarray().argmax()
    print("\n" + "=" * 80)
    print(f"similarity {test_sim2[i]:.3f} | test: {test['label'][i]} | train: {train['label'][j]}")
    print("TEST :", test["text"][i][:250])
    print("TRAIN:", train["text"][j][:250])


similarity 1.000 | test: Credit reporting | train: Credit card
TEST : This account is reporting inaccurate information on my consumer report. I have disputed this account with the company and they have failed to do a proper investigation. This account is being reported on my consumer report without my consent and causi
TRAIN: his account is reporting inaccurate information on my consumer report. I have disputed this account with the company and they have failed to do a proper investigation. This account is being reported on my consumer report without my consent and causin

similarity 1.000 | test: Debt collection | train: Credit reporting
TEST : Reporting inaccurate information on my credit report
TRAIN: Is reporting inaccurate information on my credit report.

similarity 0.995 | test: Credit reporting | train: Debt collection
TEST : I am not liable for this debt of with [REDACTED] Account # [REDACTED] I do not have a contract with [REDACTED], they did not provided me with the origina

**Finding:** The three closest remaining pairs are the same or almost the same text filed under different products (for example a complaint labelled "Credit reporting" in test and "Credit card" in train, similarity 1.000). My near-duplicate step only compared complaints within the same class, and my earlier conflict check only caught exact copies.

**What it means:** This is label noise, not leakage: the model cannot get both labels right, so it can only lower the score slightly. It is a limitation to mention in the README.

cross-label near-copies in the test set

In [14]:
idx = np.where(test_sim2 >= 0.90)[0]
S = (Xt[idx] @ X_train2.T).toarray()
j = S.argmax(axis=1)
test_labels = test["label"].values[idx]
train_labels = train["label"].values[j]
same = test_labels == train_labels

print(f"Test complaints with a >=0.90 twin in train: {len(idx)} ({len(idx) / len(test):.2%})")
print(f"  same label: {int(same.sum())} | different label: {int((~same).sum())}")
display(pd.crosstab(pd.Series(test_labels, name="test label"),
                    pd.Series(train_labels, name="train twin label")))

Test complaints with a >=0.90 twin in train: 19 (0.25%)
  same label: 0 | different label: 19


train twin label,Credit card,Credit reporting,Debt collection,Money transfer
test label,,,,
Checking or savings account,1,1,0,1
Credit card,0,1,0,0
Credit reporting,2,0,3,0
Debt collection,0,7,0,0
Student loan,0,1,0,0
Vehicle loan or lease,0,1,1,0
